# Supervised Fine-Tuning for Code Bug Detection on Azure AI Foundry

Code review is critical — but it's time-consuming, and subtle bugs slip through even experienced reviewers. Large language models can help, but the best models (like GPT-5.4) are expensive for high-volume use: **$2.50/M input tokens**, **$15/M output tokens**, and a median time-to-first-token of **181 seconds** on Azure.

In this notebook, we'll teach a smaller, cheaper model — **GPT-4.1-mini** — to match that quality using **supervised fine-tuning (SFT)**.

### What is Supervised Fine-Tuning?

SFT is the simplest form of model customization. You provide examples of correct input→output pairs, and the model learns to produce similar outputs. When the training data comes from a larger, more capable model, this process is called **distillation** — you're "distilling" the teacher's expertise into a smaller, faster student.

### What You'll Learn

1. How to evaluate a base model to establish a **baseline**
2. How to submit and monitor an SFT fine-tuning job on **Azure AI Foundry**
3. How to deploy and evaluate a **fine-tuned model**
4. How to compare **quality, cost, and latency** against the teacher model

### Notebook Roadmap

| Step | What | Tool |
|------|------|------|
| 1 | Setup & configuration | Azure OpenAI SDK |
| 2 | Inspect training data | Python |
| 3 | Baseline evaluation | Azure OpenAI (inference) |
| 4 | Upload data & submit training | Azure AI Foundry (training API) |
| 5 | Monitor training | Azure AI Foundry (events API) |
| 6 | Deploy fine-tuned model | Azure Resource Manager API |
| 7 | Evaluate & compare | Azure OpenAI (inference) |

### The Task

Given a function with a subtle bug, the model must:
- **Identify** the exact bug (not just say "there's an error")
- **Explain** why it causes incorrect behavior
- **Suggest** a minimal fix

The dataset covers 10 categories of bugs across Python, JavaScript, Java, and C++: off-by-one errors, null references, type mismatches, resource leaks, race conditions, buffer overflows, integer overflow, logic errors, unhandled exceptions, and security vulnerabilities.

### Example

**Input (buggy code):**
```python
def add_tag(posts, tag):
    posts_copy = posts.copy()
    for post in posts_copy:
        post['tags'].append(tag)
    return posts_copy
```

**Expected output:**
> **Bug**: `posts.copy()` makes a shallow copy — the inner `tags` lists are still shared with the original. Appending to `post['tags']` mutates the original posts too.
>
> **Fix**: Use `copy.deepcopy(posts)` or `[{**p, 'tags': p['tags'][:]} for p in posts]` to create independent copies of the nested lists.


## 1. Setup & Configuration

Before you can fine-tune or evaluate models, you need a few things in place. This section walks through every prerequisite so there are no surprises later.

### Prerequisites

1. **An Azure AI Foundry resource** with fine-tuning enabled. Fine-tuning is available in specific regions — check the [Azure OpenAI fine-tuning docs](https://learn.microsoft.com/azure/ai-services/openai/how-to/fine-tuning) for supported regions.

2. **Two model deployments** already created in your resource:
   - **`gpt-4.1-mini`** — the base model we'll fine-tune. This is the default deployment name; if yours differs, update `BASE_MODEL` in the code cell below.
   - **A judge model** (`gpt-5-4` or `gpt-4.1`) — used to score model responses during evaluation. GPT-5.4 gives the strictest, most reliable scores, but GPT-4.1 works as a fallback.

3. **Azure CLI** installed and authenticated (`az login`). We use it later to get access tokens for deploying the fine-tuned model.

4. **A `.env` file** with your credentials. Copy `.env.template` to `.env` and fill in:
   - `AZURE_OPENAI_ENDPOINT` — your resource endpoint (e.g., `https://my-resource.openai.azure.com/`)
   - `AZURE_OPENAI_API_KEY` — your API key (Azure Portal → your resource → Keys and Endpoint)
   - `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZURE_ACCOUNT_NAME` — needed for deploying the fine-tuned model via ARM API

5. **Install dependencies**: `pip install -r requirements.txt`


In [1]:
import json
import time
import os
import re
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

# Azure AI Foundry connection
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2025-03-01-preview")

# Azure resource details (used for deployment in Section 7)
AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
AZURE_RESOURCE_GROUP = os.environ["AZURE_RESOURCE_GROUP"]
AZURE_ACCOUNT_NAME = os.environ["AZURE_ACCOUNT_NAME"]

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Models
BASE_MODEL = "gpt-4.1-mini"
JUDGE_MODEL = "gpt-5-4"  # Fallback: "gpt-4.1" if gpt-5-4 is not deployed

print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Base model: {BASE_MODEL}")
print(f"Judge model: {JUDGE_MODEL}")
print("Setup complete!")

Endpoint: https://my-ai-foundry.openai.azure.com/
Base model: gpt-4.1-mini
Judge model: gpt-5-4
Setup complete!


## 2. Load & Inspect Training Data

Good training data is the single biggest factor in fine-tuning success. Let's look at what we're working with.

### About the Dataset

- **224 training examples** + **20 validation examples** of buggy code paired with expert-quality analysis
- Each example covers a different bug across **10 categories** (off-by-one errors, null references, type mismatches, resource leaks, race conditions, buffer overflows, integer overflow, logic errors, unhandled exceptions, and security vulnerabilities) in **multiple languages** (Python, JavaScript, Java, C++)
- This is a **distillation dataset** — the training examples were generated by GPT-5.4 (the teacher model) and then quality-filtered. Only examples scoring **≥7/10** on a quality rubric were kept. This curation step is critical: in our experiments, 224 high-quality examples outperformed 1,576 noisy ones.

### The SFT Chat Format

Each training example is a JSON object with a `messages` array in the standard chat format:

| Role | Purpose |
|------|------|
| **system** | Instructions for the bug detection task — tells the model what kind of response to produce |
| **user** | A code snippet containing a subtle bug |
| **assistant** | The ideal response — identifies the bug, explains why it's wrong, and suggests a minimal fix |

During fine-tuning, the model learns to generate the **assistant** message given the system + user context. That's the core of SFT: learning by example.


In [2]:
DATA_DIR = os.path.join("..", "..", "Sample_Datasets", "Supervised_Fine_Tuning", "Text-Bug-Detection")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

training_data = load_jsonl(os.path.join(DATA_DIR, "training_data.jsonl"))
validation_data = load_jsonl(os.path.join(DATA_DIR, "validation_data.jsonl"))

print(f"Training examples: {len(training_data)}")
print(f"Validation examples: {len(validation_data)}")

# Show one example
example = training_data[0]
msgs = example["messages"]
print("\n--- Example ---")
print(f"System: {msgs[0]['content'][:120]}...")
print(f"User:   {msgs[1]['content'][:120]}...")
print(f"Asst:   {msgs[2]['content'][:120]}...")

Training examples: 224
Validation examples: 20

--- Example ---
System: You are an expert software engineer. Analyze the following code for bugs. Identify the bug, explain why it ca...
User:   The following Python code is supposed to merge two sorted lists into a single sorted list, but it contains a b...
Asst:   ## Bug Identification
The bug is in the merge loop's comparison logic. The code...


## 3. Baseline Evaluation

Before fine-tuning, we need to know how well the base model already performs. This is your **control group** — without it, you can't measure whether fine-tuning actually helped. (Sometimes the base model is already good enough, and fine-tuning would be wasted effort!)

### Evaluation Approach

1. **Run** each test example through the base GPT-4.1-mini model
2. **Score** each response using a judge model (GPT-5.4) on three dimensions:
   - **Bug identification** (1–10): Did the model correctly find the bug?
   - **Explanation quality** (1–10): Is the explanation clear, accurate, and complete?
   - **Fix quality** (1–10): Is the proposed fix correct and following best practices?
3. **Pass/fail**: A combined score ≥ 7.0 counts as a "pass"

We use the **same judge model and rubric** for every evaluation in this notebook — the baseline, the fine-tuned model, and the teacher. Consistency is key: without a fixed rubric, you can't compare results across runs.

The next two code cells define the helper functions (`get_completion`, `judge_response`, `evaluate_model`) and then run the baseline evaluation.


In [3]:
# Prepare test examples (use validation set, cap at 10 for cost efficiency)
test_examples = validation_data[:10]
print(f"Using {len(test_examples)} test examples for evaluation")


def get_completion(model, messages, max_completion_tokens=1024):
    """Get a chat completion from the given model."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        max_completion_tokens=max_completion_tokens,
    )
    return response.choices[0].message.content


def judge_response(user_input, model_response, reference_response):
    """Use the judge model to score a bug detection response."""
    rubric = (
        "You are an expert code reviewer judging bug detection responses.\n"
        "Score the candidate response compared to the reference on three criteria, each 1-10:\n"
        "- bug_identification: Did the candidate correctly identify the bug?\n"
        "- explanation: Is the explanation clear, accurate, and include a failing test case?\n"
        "- fix_quality: Is the proposed fix correct and minimal?\n\n"
        "Return ONLY a JSON object with these three keys and integer scores.\n"
        'Example: {"bug_identification": 8, "explanation": 7, "fix_quality": 9}'
    )

    prompt = (
        "## Buggy Code\n" + user_input + "\n\n"
        "## Reference Answer\n" + reference_response + "\n\n"
        "## Candidate Answer\n" + model_response + "\n\n"
        "Score the candidate answer. Return ONLY the JSON object."
    )

    judge_messages = [
        {"role": "system", "content": rubric},
        {"role": "user", "content": prompt},
    ]

    raw = get_completion(JUDGE_MODEL, judge_messages, max_completion_tokens=256)
    return parse_judge_scores(raw)


def parse_judge_scores(raw):
    """Extract JSON scores from the judge response."""
    # Strip markdown code fences if present
    cleaned = re.sub(r"```json\s*", "", raw)
    cleaned = re.sub(r"```\s*", "", cleaned)
    cleaned = cleaned.strip()
    try:
        scores = json.loads(cleaned)
        return scores
    except json.JSONDecodeError:
        print(f"  Warning: Could not parse judge response: {raw[:100]}")
        return {"bug_identification": 0, "explanation": 0, "fix_quality": 0}

Using 10 test examples for evaluation


In [4]:
def evaluate_model(model_name, test_examples, label="Model"):
    """Run test examples through a model and judge the responses."""
    results = []
    for i, example in enumerate(test_examples):
        msgs = example["messages"]
        system_msg = msgs[0]["content"]
        user_msg = msgs[1]["content"]
        reference = msgs[2]["content"]

        # Get model response
        response = get_completion(
            model_name,
            [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
        )
        time.sleep(0.5)

        # Judge the response
        scores = judge_response(user_msg, response, reference)
        time.sleep(0.5)

        combined = (
            scores.get("bug_identification", 0)
            + scores.get("explanation", 0)
            + scores.get("fix_quality", 0)
        ) / 3.0

        results.append({"scores": scores, "combined": combined})
        print(
            f"  [{label}] Example {i+1}: "
            f"bug={scores.get('bug_identification', 0)} "
            f"expl={scores.get('explanation', 0)} "
            f"fix={scores.get('fix_quality', 0)} "
            f"combined={combined:.1f}"
        )

    # Summary
    avg_combined = sum(r["combined"] for r in results) / len(results)
    pass_count = sum(1 for r in results if r["combined"] >= 7.0)
    pass_rate = pass_count / len(results) * 100

    print(f"\n  {label} Summary:")
    print(f"    Average combined score: {avg_combined:.2f}")
    print(f"    Pass rate (>=7.0): {pass_count}/{len(results)} ({pass_rate:.1f}%)")

    return {"results": results, "avg_combined": avg_combined, "pass_rate": pass_rate}


print("Evaluating base model (gpt-4.1-mini)...\n")
baseline_eval = evaluate_model(BASE_MODEL, test_examples, label="Baseline")

Evaluating base model (gpt-4.1-mini)...

  [Baseline] Example 1: bug=10 expl=10 fix=10 combined=10.0
  [Baseline] Example 2: bug=9 expl=8 fix=9 combined=8.7
  [Baseline] Example 3: bug=8 expl=7 fix=7 combined=7.3
  [Baseline] Example 4: bug=10 expl=10 fix=9 combined=9.7
  [Baseline] Example 5: bug=10 expl=10 fix=10 combined=10.0
  [Baseline] Example 6: bug=10 expl=10 fix=10 combined=10.0
  [Baseline] Example 7: bug=10 expl=10 fix=10 combined=10.0
  [Baseline] Example 8: bug=2 expl=2 fix=1 combined=1.7
  [Baseline] Example 9: bug=10 expl=10 fix=10 combined=10.0
  [Baseline] Example 10: bug=7 expl=5 fix=5 combined=5.7

  Baseline Summary:
    Average combined score: 8.31
    Pass rate (>=7.0): 8/10 (80.0%)


## 4. Upload Training Data

Azure AI Foundry needs your training data uploaded as files before you can reference them in a fine-tuning job. This is a two-step process:

1. **Upload** the JSONL files using the Files API
2. **Wait** for the platform to validate and process them

The platform validates the JSONL format during upload — if there are schema issues (missing `messages` field, wrong role names, encoding errors), you'll get errors here rather than during training. This is a good thing: it's much faster to fix a data issue than to wait for a training job to fail.


In [5]:
print("Uploading training file...")
with open(os.path.join(DATA_DIR, "training_data.jsonl"), "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
print(f"Training file ID: {train_file.id}")

print("\nUploading validation file...")
with open(os.path.join(DATA_DIR, "validation_data.jsonl"), "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")
print(f"Validation file ID: {val_file.id}")

# Wait for file processing
print("\nWaiting for files to be processed...")
for file_id in [train_file.id, val_file.id]:
    while True:
        file_info = client.files.retrieve(file_id)
        if file_info.status == "processed":
            break
        print(f"  File {file_id}: status={file_info.status}, waiting...")
        time.sleep(5)

print("Files ready!")

Uploading training file...
Training file ID: file-abc123example

Uploading validation file...
Validation file ID: file-def456example

Waiting for files to be processed...
Files ready!


## 5. Submit Fine-Tuning Job

Now we submit the training job. SFT has three main hyperparameters — here's what each one does and why we chose these values:

- **`n_epochs`** (2): The number of complete passes through the training dataset. More epochs means the model sees each example more times, which can improve learning — but too many epochs leads to **overfitting** (the model memorizes the training data instead of learning general patterns). We use 2 because our experiments showed that 3 epochs caused the validation loss to rise 74% above its best value, a clear sign of overfitting on this dataset size.

- **`learning_rate_multiplier`** (0.8): Controls how aggressively the model updates its weights on each training step. Higher values mean faster learning but less stability; lower values are safer but may underutilize the data. We use 0.8 (slightly conservative) because higher values like 2.0 caused training instability in our experiments, while lower values like 0.5 produced weaker results.

- **`suffix`** ("bug-detection"): A name tag appended to your fine-tuned model ID. It doesn't affect training — it just makes the model easy to find later in your list of fine-tuned models.

With 224 examples and 2 epochs, training typically takes **15–30 minutes**.


In [6]:
print("Submitting fine-tuning job...")

ft_job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=val_file.id,
    hyperparameters={
        "n_epochs": 2,
        "learning_rate_multiplier": 0.8,
    },
    suffix="bug-detection",
)

print(f"Job ID: {ft_job.id}")
print(f"Status: {ft_job.status}")
print(f"Model: {ft_job.model}")

Submitting fine-tuning job...
Job ID: ftjob-example123
Status: pending
Model: gpt-4.1-mini


## 6. Monitor Training

The cell below polls the training job every 30 seconds and prints events as they arrive. The most important thing to watch is the **training loss** — it should decrease over time, meaning the model is learning from the data.

### What Healthy Training Looks Like

- ✅ **Healthy**: Loss decreases steadily over training steps. Validation loss tracks training loss (with a small gap).
- ⚠️ **Warning — overfitting**: Training loss keeps dropping but validation loss starts *increasing*. The model is memorizing rather than generalizing. Try fewer epochs or more training data.
- ⚠️ **Warning — instability**: Loss spikes or explodes in later steps. The learning rate may be too high — try a lower `learning_rate_multiplier`.
- ⚠️ **Warning — plateau**: Loss stops decreasing early. The model may need more data, more epochs, or a higher learning rate.

You can also monitor training in the [Azure AI Foundry portal](https://ai.azure.com) under **Build → Fine-tuning**, which provides a graphical loss curve.


In [7]:
print(f"Monitoring job {ft_job.id}...\n")

seen_events = set()

while True:
    job = client.fine_tuning.jobs.retrieve(ft_job.id)

    # Print new events
    events = client.fine_tuning.jobs.list_events(ft_job.id, limit=50)
    for event in events.data:
        if event.id not in seen_events:
            seen_events.add(event.id)
            print(f"  [{event.created_at}] {event.message}")

    print(f"  Status: {job.status}")

    if job.status in ("succeeded", "failed", "cancelled"):
        break

    time.sleep(30)

if job.status == "succeeded":
    FINE_TUNED_MODEL = job.fine_tuned_model
    print(f"\nFine-tuned model: {FINE_TUNED_MODEL}")
else:
    print(f"\nJob ended with status: {job.status}")

Monitoring job ftjob-example123...

  [1751234560] Validating training file: file-abc123example
  [1751234562] Validating validation file: file-def456example
  [1751234565] Files validated successfully
  Status: validating_files
  [1751234600] Training file preprocessing complete: 224 examples, ~85K tokens
  [1751234602] Validation file preprocessing complete: 20 examples, ~7.5K tokens
  [1751234605] Fine-tuning job started. Training for 2 epochs (448 steps)
  Status: running
  [1751234700] Step 50/448 — training loss: 0.8513
  Status: running
  [1751234800] Step 100/448 — training loss: 0.5184
  [1751234810] Step 100/448 — validation loss: 0.6231
  Status: running
  [1751234900] Step 150/448 — training loss: 0.4127
  Status: running
  [1751235000] Step 200/448 — training loss: 0.3491
  [1751235010] Step 200/448 — validation loss: 0.4102
  Status: running
  [1751235100] Step 250/448 — training loss: 0.2843
  Status: running
  [1751235200] Step 300/448 — training loss: 0.2297
  [1751235

## 7. Deploy the Fine-Tuned Model

Fine-tuned models aren't automatically available for inference — you need to **deploy** them first. This creates an endpoint you can call just like any other Azure OpenAI model.

We deploy using the Azure Resource Manager (ARM) REST API, which gives us programmatic control over the deployment configuration. Here are the key settings:

- **SKU: `GlobalStandard`** — this is required for fine-tuned OpenAI models. (Don't use `Standard` — it won't work with custom models.)
- **Capacity: 100** — tokens per minute, in thousands. This is enough for evaluation; scale up for production workloads.
- **Warmup time: ~5 minutes** — new deployments need a few minutes before they respond reliably. Calling too early gives `BadRequestForDependentService` errors, so we build in a wait.

After evaluation, we'll delete this deployment to avoid unnecessary hosting costs.


In [8]:
import subprocess
import requests

DEPLOYMENT_NAME = "bug-detection-ft"

# Find Azure CLI
az_cmd = None
for candidate in ["az", r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"]:
    try:
        subprocess.run([candidate, "--version"], capture_output=True, check=True)
        az_cmd = candidate
        break
    except (FileNotFoundError, subprocess.CalledProcessError):
        continue

if az_cmd is None:
    raise RuntimeError("Azure CLI not found. Install it from https://aka.ms/installazurecli")

print(f"Using Azure CLI: {az_cmd}")

# Get access token
token_result = subprocess.run(
    [az_cmd, "account", "get-access-token", "--query", "accessToken", "-o", "tsv"],
    capture_output=True, text=True, check=True,
)
access_token = token_result.stdout.strip()

# Deploy via ARM REST API
url = (
    f"https://management.azure.com/subscriptions/{AZURE_SUBSCRIPTION_ID}"
    f"/resourceGroups/{AZURE_RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{AZURE_ACCOUNT_NAME}"
    f"/deployments/{DEPLOYMENT_NAME}?api-version=2023-05-01"
)

body = {
    "sku": {"name": "GlobalStandard", "capacity": 100},
    "properties": {
        "model": {
            "format": "OpenAI",
            "name": FINE_TUNED_MODEL,
            "version": "1",
        }
    },
}

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

print(f"Deploying {FINE_TUNED_MODEL} as '{DEPLOYMENT_NAME}'...")
resp = requests.put(url, headers=headers, json=body)
resp.raise_for_status()
print(f"Deployment response: {resp.status_code}")
print(resp.json().get("properties", {}).get("provisioningState", "unknown"))

Using Azure CLI: az
Deploying gpt-4.1-mini-2025-04-14.ft-example123-bug-detection as 'bug-detection-ft'...
Deployment response: 201
Succeeded


In [9]:
# Wait for the deployment to warm up
# New fine-tuned model deployments need a few minutes to become responsive.
WARMUP_SECONDS = 300
print(f"Waiting {WARMUP_SECONDS // 60} minutes for deployment to warm up...")
time.sleep(WARMUP_SECONDS)
print("Deployment should be ready!")

Waiting 5 minutes for deployment to warm up...
Deployment should be ready!


## 8. Evaluate the Fine-Tuned Model

The key question: **did fine-tuning help?**

We run the exact same test examples from Section 3 through the fine-tuned model, scored by the same judge with the same rubric. This apples-to-apples comparison tells us whether the training data actually improved the model's bug detection ability.

If the fine-tuned model's pass rate and average score are higher than the baseline, we know SFT worked. If they're the same or worse, we'd need to revisit our training data quality, hyperparameters, or dataset size.


In [10]:
print("Evaluating fine-tuned model...\n")
ft_eval = evaluate_model(DEPLOYMENT_NAME, test_examples, label="Fine-Tuned")

print("\n--- Baseline vs Fine-Tuned ---")
print(f"  {'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>10}")
print(f"  {'-'*25} {'-'*10} {'-'*10}")
print(f"  {'Combined Score':<25} {baseline_eval['avg_combined']:>10.2f} {ft_eval['avg_combined']:>10.2f}")
print(f"  {'Pass Rate (>=7.0)':<25} {baseline_eval['pass_rate']:>9.1f}% {ft_eval['pass_rate']:>9.1f}%")

Evaluating fine-tuned model...

  [Fine-Tuned] Example 1: bug=10 expl=10 fix=10 combined=10.0
  [Fine-Tuned] Example 2: bug=10 expl=10 fix=9 combined=9.7
  [Fine-Tuned] Example 3: bug=9 expl=9 fix=9 combined=9.0
  [Fine-Tuned] Example 4: bug=9 expl=8 fix=9 combined=8.7
  [Fine-Tuned] Example 5: bug=10 expl=10 fix=10 combined=10.0
  [Fine-Tuned] Example 6: bug=10 expl=10 fix=10 combined=10.0
  [Fine-Tuned] Example 7: bug=10 expl=9 fix=9 combined=9.3
  [Fine-Tuned] Example 8: bug=6 expl=5 fix=5 combined=5.3
  [Fine-Tuned] Example 9: bug=10 expl=9 fix=10 combined=9.7
  [Fine-Tuned] Example 10: bug=10 expl=10 fix=10 combined=10.0

  Fine-Tuned Summary:
    Average combined score: 9.17
    Pass rate (>=7.0): 9/10 (90.0%)

--- Baseline vs Fine-Tuned ---
  Metric                      Baseline Fine-Tuned
  ------------------------- ---------- ----------
  Combined Score                  8.31       9.17
  Pass Rate (>=7.0)              80.0%      90.0%


## 9. Compare to the Teacher Model

Our final comparison: how does the fine-tuned mini model stack up against **GPT-5.4** — the teacher that generated the training data?

This is the payoff of distillation. If the student matches or approaches the teacher's quality, we've successfully transferred that capability at **much lower cost and latency**. And if the student *beats* the teacher on some metrics (which happens more often than you'd expect), it's because fine-tuning teaches a consistent, focused output style that the larger model doesn't naturally produce.

We evaluate GPT-5.4 on the same test set with the same judge, then display all three models side by side — including cost and latency comparisons.


In [11]:
print("Evaluating teacher model (GPT-5.4)...\n")
teacher_eval = evaluate_model(JUDGE_MODEL, test_examples, label="Teacher")

Evaluating teacher model (GPT-5.4)...

  [Teacher] Example 1: bug=10 expl=10 fix=9 combined=9.7
  [Teacher] Example 2: bug=10 expl=10 fix=10 combined=10.0
  [Teacher] Example 3: bug=10 expl=10 fix=10 combined=10.0
  [Teacher] Example 4: bug=10 expl=10 fix=10 combined=10.0
  [Teacher] Example 5: bug=10 expl=10 fix=10 combined=10.0
  [Teacher] Example 6: bug=10 expl=10 fix=10 combined=10.0
  [Teacher] Example 7: bug=8 expl=8 fix=8 combined=8.0
  [Teacher] Example 8: bug=5 expl=4 fix=4 combined=4.3
  [Teacher] Example 9: bug=10 expl=10 fix=10 combined=10.0
  [Teacher] Example 10: bug=7 expl=6 fix=6 combined=6.3

  Teacher Summary:
    Average combined score: 8.83
    Pass rate (>=7.0): 8/10 (80.0%)


In [12]:
# Final comparison table
print("\n" + "=" * 110)
print("FINAL COMPARISON")
print("=" * 110)

# Verified pricing (per 1M tokens) and TTFT from artificialanalysis.ai
model_info = {
    "gpt-4.1-mini (base)": {
        "combined": baseline_eval["avg_combined"],
        "pass_rate": baseline_eval["pass_rate"],
        "input_cost": 0.40,
        "output_cost": 1.60,
        "ttft": 1.35,
    },
    "gpt-4.1-mini FT": {
        "combined": ft_eval["avg_combined"],
        "pass_rate": ft_eval["pass_rate"],
        "input_cost": 0.40,
        "output_cost": 1.60,
        "ttft": 1.35,
    },
    "gpt-5.4 (teacher)": {
        "combined": teacher_eval["avg_combined"],
        "pass_rate": teacher_eval["pass_rate"],
        "input_cost": 2.50,
        "output_cost": 15.00,
        "ttft": 181.2,
    },
}

teacher = model_info["gpt-5.4 (teacher)"]

header = (
    f"{'Model':<22} {'Combined':>8} {'Pass@7':>7} "
    f"{'In $/1M':>8} {'Out $/1M':>9} {'TTFT(s)':>8} "
    f"{'vs Teacher Q':>13} {'vs Teacher $':>13} {'vs Teacher Lat':>15}"
)
print(header)
print("-" * 110)

for name, info in model_info.items():
    # Compute vs-teacher metrics
    q_ratio = info["combined"] / teacher["combined"] * 100 if teacher["combined"] > 0 else 0
    avg_cost = (info["input_cost"] + info["output_cost"]) / 2
    teacher_avg_cost = (teacher["input_cost"] + teacher["output_cost"]) / 2
    cost_ratio = avg_cost / teacher_avg_cost * 100 if teacher_avg_cost > 0 else 0
    lat_ratio = info["ttft"] / teacher["ttft"] * 100 if teacher["ttft"] > 0 else 0

    row = (
        f"{name:<22} {info['combined']:>8.2f} {info['pass_rate']:>6.1f}% "
        f"${info['input_cost']:>6.2f} ${info['output_cost']:>7.2f} {info['ttft']:>8.2f} "
        f"{q_ratio:>12.1f}% {cost_ratio:>12.1f}% {lat_ratio:>14.1f}%"
    )
    print(row)

print("=" * 110)
print("\nKey takeaway: The fine-tuned mini model delivers teacher-level quality")
print("at a fraction of the cost and latency.")


FINAL COMPARISON
Model                  Combined  Pass@7  In $/1M  Out $/1M  TTFT(s)  vs Teacher Q  vs Teacher $  vs Teacher Lat
--------------------------------------------------------------------------------------------------------------
gpt-4.1-mini (base)        8.31   80.0% $  0.40 $   1.60     1.35         94.1%         11.4%            0.7%
gpt-4.1-mini FT            9.17   90.0% $  0.40 $   1.60     1.35        103.9%         11.4%            0.7%
gpt-5.4 (teacher)          8.83   80.0% $  2.50 $  15.00   181.20        100.0%        100.0%          100.0%

Key takeaway: The fine-tuned mini model delivers teacher-level quality
at a fraction of the cost and latency.


## 10. Cleanup

Delete the deployment to stop incurring hosting costs. The fine-tuned **model weights are retained** — you can redeploy anytime without retraining.

**💡 Tip:** For evaluation workflows, use **Developer Tier** deployments — they're free to host and are automatically deleted after 24 hours.


In [13]:
# Delete the fine-tuned model deployment
delete_url = (
    f"https://management.azure.com/subscriptions/{AZURE_SUBSCRIPTION_ID}"
    f"/resourceGroups/{AZURE_RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{AZURE_ACCOUNT_NAME}"
    f"/deployments/{DEPLOYMENT_NAME}?api-version=2023-05-01"
)

# Refresh the access token in case the old one expired
token_result = subprocess.run(
    [az_cmd, "account", "get-access-token", "--query", "accessToken", "-o", "tsv"],
    capture_output=True, text=True, check=True,
)
access_token = token_result.stdout.strip()
headers = {"Authorization": f"Bearer {access_token}"}

print(f"Deleting deployment '{DEPLOYMENT_NAME}'...")
resp = requests.delete(delete_url, headers=headers)
print(f"Status: {resp.status_code}")

if resp.status_code in (200, 202, 204):
    print("Deployment deleted successfully.")
else:
    print(f"Delete response: {resp.text}")

# To clean up training files, uncomment:
# client.files.delete(train_file.id)
# client.files.delete(val_file.id)
# print("Training files deleted.")

print("\nDone! Cleanup complete.")

Deleting deployment 'bug-detection-ft'...
Status: 204
Deployment deleted successfully.

Done! Cleanup complete.
